# Feature Engineering done using pipe -  Imputer_Discretizer_Rarelablremov_encode_scaler

## 1. Import libaries

In [9]:
from math import sqrt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline as pipe
from sklearn.preprocessing import MinMaxScaler

from feature_engine.encoding import RareLabelEncoder, MeanEncoder
from feature_engine.discretisation import DecisionTreeDiscretiser
from feature_engine.imputation import (
    AddMissingIndicator,
    MeanMedianImputer,
    CategoricalImputer,
    )

## 2. Set Data

In [11]:
# Load dataset
data = pd.read_csv('./Dataset/houseprice.csv')

# drop some variables
data.drop(
    labels=['YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'Id'],
    axis=1,
    inplace=True
    )

# Separate into train and test sets
# separate into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
                                        data.drop(labels=['SalePrice'], axis=1),
                                        data.SalePrice,
                                        test_size=0.1,
                                        random_state=0
                                        )

In [19]:
# make a list of categorical variables
categorical = [var for var in data.columns if data[var].dtype == 'O']

# make a list of numerical variables
numerical = [var for var in data.columns if data[var].dtype != 'O']

# make a list of discrete variables (It includes both dtype == 'O' AND NUMERIC ALSO)
discrete = [ var for var in numerical if len(data[var].unique()) < 20]

# Convert all discrete variable(unique< 20) to categorical

# categorical encoders work only with object type variables
# to treat numerical variables as categorical, we need to re-cast them
data[discrete]= data[discrete].astype('O')

# continuous variables, Excluding all discrete variable
numerical = [
    var for var in numerical if var not in discrete
    and var not in ['Id', 'SalePrice']
    ]

## 3. Set up the pipeline

##### 1. Imputer

- continuous_var_imputer for indicator -> AddMissingIndicator
- continuous_var_median_imputer -> MeanMedianImputer 
- categorical_imputer -> CategoricalImputer


##### 2. Discretize all continous/Numerical value TO Discrete value

- numerical_tree_discretiser  - > DecisionTreeDiscretiser

##### 3. Encoder

###### 3.1 rare_label_encoder

  - RareLabelEncoder -> remove rare labels in categorical and discrete variables
  
###### 3.2 categorical_encoder

  - categorical_encoder -> encode categorical and discrete variables using the target mean
  
#####  4. Scaler

- MinMaxScaler -> Apply MinMaxScalerization

##### 5. Model Build: lasso

- lasso -> Apply lasso Regression

In [38]:
price_pipe = pipe([
    # add a binary variable to indicate missing information for the 2 variables below
    ('continuous_var_imputer', AddMissingIndicator(variables=['LotFrontage'])),

    # replace NA by the median in the 2 variables below, they are numerical
    ('continuous_var_median_imputer', MeanMedianImputer(
        imputation_method='median', variables=['LotFrontage', 'MasVnrArea']
    )),

    # replace NA by adding the label "Missing" in categorical variables
    ('categorical_imputer', CategoricalImputer(variables=categorical)),

    # disretise continuous variables using trees
    ('numerical_tree_discretiser', DecisionTreeDiscretiser(
        cv=3,
        scoring='neg_mean_squared_error',
        variables=numerical,
        regression=True)),

    # remove rare labels in categorical and discrete variables
    ('rare_label_encoder', RareLabelEncoder(
        tol=0.03, n_categories=1, variables=categorical+discrete
    )),

    # encode categorical and discrete variables using the target mean
    ('categorical_encoder', MeanEncoder(variables=categorical+discrete)),

    # scale features
    ('scaler', MinMaxScaler()),

    # Lasso
    ('lasso', Lasso(random_state=2909, alpha=0.005))

])


## 4. Build Model

In [37]:
# train feature engineering transformers and Lasso
price_pipe.fit(X_train, np.log(y_train))

# predict
pred_train = price_pipe.predict(X_train)
pred_test = price_pipe.predict(X_test)

TypeError: Some of the variables are not categorical. Please cast them as object or category before calling this transformer

In [ ]:
## 5. Evaluate Model

In [16]:
print('Lasso Linear Model train mse: {}'.format(
    mean_squared_error(y_train, np.exp(pred_train))))
print('Lasso Linear Model train rmse: {}'.format(
    sqrt(mean_squared_error(y_train, np.exp(pred_train)))))
print()
print('Lasso Linear Model test mse: {}'.format(
    mean_squared_error(y_test, np.exp(pred_test))))
print('Lasso Linear Model test rmse: {}'.format(
    sqrt(mean_squared_error(y_test, np.exp(pred_test)))))

NameError: name 'pred_train' is not defined

In [ ]:
plt.scatter(y_test, np.exp(pred_test))
plt.xlabel('True Price')
plt.ylabel('Predicted Price')
plt.show()